[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/06_polyphonic_mert_multilabel.ipynb)


# 06 - Polyphonic MERT Multi-Label Classifier

## Research question

How well does the selected isolated-stem MERT representation transfer to polyphonic multi-label instrument recognition?

## Approach

The multi-label configs train a sigmoid/BCE head on mixture embeddings and save global metrics, per-class metrics, predictions, resolved config, checkpoints, and registry rows. Swap `representation` and `data.cache_dir` in YAML if notebooks 01-04 identify a better isolated-stem representation.

## What is fixed

Unless a selected config says otherwise: MedleyDB instrument labels, the `largest_balanced` split, MERT-v1-95M, 5 s segments, validation-based model selection, and test metrics saved only after training.

## What is varied

Synthetic controlled-overlap mixtures, same-song same-time reconstructed mixtures, and original full-mix segments.

## Expected interpretation

Compare synthetic and realistic protocols separately; do not collapse them unless the report explains the different label-generation assumptions.


## What you can change

- `PROJECT_ROOT`, `RUN_ROOT`, and `MEDLEYDB_ROOT` for local vs Colab/Drive paths.
- `EXPERIMENT_GROUPS`, `DATASET_GROUPS`, `MIXTURE_GROUPS`, and `SELECTED_EXPERIMENTS` to run one config at a time.
- `REPLACE_EXISTING = True` only when intentionally overwriting a completed run.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR, RUN_ROOT / "configs"]:
    path.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)


## Representation Note

The polyphonic model should normally use the best isolated-stem MERT configuration selected from notebooks 01-04. The default configs are fixed to MERT-v1-95M last-layer mean pooling so the protocol is reproducible, but the cache path and representation fields are intentionally simple to swap.

In [ ]:
def q(path):
    return shlex.quote(str(path))


def run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)


def load_config(config_path):
    return yaml.safe_load(Path(config_path).read_text(encoding="utf-8"))


def run_experiment_config(config_path, *, extract_embeddings=True, replace_existing=False):
    config_path = Path(config_path)
    cfg = load_config(config_path)
    approach = cfg.get("approach", "frozen_embeddings")
    if approach == "frozen_embeddings" and extract_embeddings:
        run(f"python -m src.features.extract_mert_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    elif approach == "polyphonic_multilabel" and extract_embeddings:
        run(f"python -m src.features.extract_mert_mixture_embeddings --experiment-config {q(config_path)} --batch-size 1 --device auto")
    args = f"--config {q(config_path)}"
    if replace_existing:
        args += " --replace-existing"
    return run(f"python -m src.experiments.run_experiment {args}")


def run_selected(configs, *, extract_embeddings=True, replace_existing=False):
    for config in configs:
        run_experiment_config(config, extract_embeddings=extract_embeddings, replace_existing=replace_existing)


In [ ]:
EXPERIMENT_GROUPS = {
    "polyphonic_protocols": [
        "configs/experiments/polyphonic_largest_balanced_synthetic_k_mert95_last_mean_mlp.yaml",
        "configs/experiments/polyphonic_largest_balanced_same_song_same_time_mert95_last_mean_mlp.yaml",
        "configs/experiments/polyphonic_largest_balanced_full_mix_mert95_last_mean_mlp.yaml",
    ],
}
SELECTED_EXPERIMENTS = [
    "configs/experiments/polyphonic_largest_balanced_synthetic_k_mert95_last_mean_mlp.yaml",
]
REPLACE_EXISTING = False

run_selected(SELECTED_EXPERIMENTS, extract_embeddings=True, replace_existing=REPLACE_EXISTING)
